# Stitch3D — Export Anny body GLBs

Generates the three Anny human meshes used by the **See It Worn** showcase:

| File | Preset |
|------|--------|
| `adult-neutral.glb` | Balanced adult |
| `athletic-male.glb` | Athletic male |
| `young-female.glb` | Young adult female |

**After export:** download the ZIP and copy the `.glb` files into your project at:

```
public/models/anny/
```

Source: [naver/anny](https://github.com/naver/anny) (Apache 2.0) · MakeHuman assets CC0

## 1. Install dependencies

Anny requires the **`[warp]`** extra (NVIDIA Warp). Restart runtime if Colab asks.

In [ ]:
!pip install -q "anny[warp]" trimesh torch roma

## 2. Imports & configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
import zipfile

import anny
import roma
import torch
import trimesh

# Rotate Z-up (Anny) → Y-up (Three.js / glTF) — same as official Gradio demo.
VIEW_TRANSFORM = roma.Rigid(
    roma.euler_to_rotmat("x", [-90.0], degrees=True),
    torch.zeros(3),
).to_homogeneous().numpy()

# Phenotype sliders (0–1). Must match src/canvas/AnnyFigure.jsx ids.
FIGURE_PRESETS = {
    "adult-neutral": {
        "age": 0.55,
        "gender": 0.5,
        "weight": 0.5,
        "muscle": 0.5,
        "height": 0.5,
    },
    "athletic-male": {
        "age": 0.45,
        "gender": 0.0,
        "weight": 0.42,
        "muscle": 0.78,
        "height": 0.62,
    },
    "young-female": {
        "age": 0.72,
        "gender": 1.0,
        "weight": 0.46,
        "muscle": 0.38,
        "height": 0.48,
    },
}

OUT_DIR = Path("/content/anny_glb")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output folder: {OUT_DIR}")
print(f"Presets: {list(FIGURE_PRESETS.keys())}")

## 3. Export function

In [ ]:
def export_glb(output_path: Path, phenotype_overrides: dict[str, float]) -> None:
    """Build one Anny full-body mesh and save as GLB."""
    dtype = torch.float32
    model = anny.create_fullbody_model(
        rig="default",
        topology="default",
        local_changes="default",
        extrapolate_phenotypes=False,
        bone_orientation="blender-rootidentity",
    ).to(dtype=dtype)

    phenotype_kwargs = {key: 0.5 for key in model.phenotype_labels}
    for key, value in phenotype_overrides.items():
        if key in phenotype_kwargs:
            phenotype_kwargs[key] = value

    local_changes_kwargs = {key: 0.0 for key in model.local_change_labels}
    bones_rotvec = torch.zeros((len(model.bone_labels), 3), dtype=dtype)

    bones_rotmat = roma.rotvec_to_rotmat(torch.deg2rad(bones_rotvec))
    translations = torch.zeros((len(bones_rotmat), 3), dtype=dtype)
    pose_parameters = roma.Rigid(bones_rotmat, translations)[None].to_homogeneous()

    output = model(
        pose_parameters=pose_parameters,
        phenotype_kwargs=phenotype_kwargs,
        local_changes_kwargs=local_changes_kwargs,
    )

    vertices = output["vertices"].squeeze(0).cpu().numpy()
    faces = model.faces.cpu().numpy()

    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    mesh.visual = trimesh.visual.TextureVisuals(
        material=trimesh.visual.material.PBRMaterial(
            baseColorFactor=[0.82, 0.84, 0.88, 0.45],
            metallicFactor=0.15,
            roughnessFactor=0.55,
            doubleSided=True,
            alphaMode="BLEND",
        )
    )

    scene = trimesh.Scene()
    scene.add_geometry(mesh, node_name="AnnyBody")
    scene.apply_transform(VIEW_TRANSFORM)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    scene.export(str(output_path))
    print(f"Exported {output_path.name} ({output_path.stat().st_size / 1024:.0f} KB)")

## 4. Run export (all 3 bodies)

First run downloads Anny model data (~cache in `~/.cache/anny/`). Expect ~30–60 s on CPU.

In [ ]:
for name, preset in FIGURE_PRESETS.items():
    export_glb(OUT_DIR / f"{name}.glb", preset)

print("\nDone. Files:")
for p in sorted(OUT_DIR.glob("*.glb")):
    print(f"  {p.name}")

## 5. Download as ZIP

Unzip and place the three `.glb` files in `public/models/anny/` in your Stitch3D repo.

In [ ]:
from google.colab import files

zip_path = Path("/content/anny_glb.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for glb in sorted(OUT_DIR.glob("*.glb")):
        zf.write(glb, arcname=glb.name)

print(f"Created {zip_path} ({zip_path.stat().st_size / 1024:.0f} KB)")
files.download(str(zip_path))

---

### Optional: export a custom body

Tweak phenotype values (0 = min, 1 = max) and re-run this cell only.

In [ ]:
custom_preset = {
    "age": 0.5,
    "gender": 0.5,
    "weight": 0.6,
    "muscle": 0.7,
    "height": 0.55,
}

export_glb(OUT_DIR / "custom-body.glb", custom_preset)
files.download(str(OUT_DIR / "custom-body.glb"))